<a href="https://colab.research.google.com/github/IanWen103026/vocabuddy-group-4/blob/main/VocaBuddy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install httpx opencc-python-reimplemented -q

import sys
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

import os
import re
import csv
import io
import time
import random
import getpass
from difflib import SequenceMatcher
from typing import List, Dict, Tuple, Optional, Any
import httpx

try:
    from opencc import OpenCC
    cc = OpenCC('s2t')
except Exception:
    cc = None

print("[OK] 套件載入成功！")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.8/481.8 kB 7.3 MB/s eta 0:00:00
[OK] 套件載入成功！


In [2]:
class VocabularyManager:
    """單字庫管理核心：支援分類、手動新增/刪除、CSV 匯入匯出與統計"""

    def __init__(self):
        # 資料結構: { category: { english: { 'chinese': str, 'correct': int, 'wrong': int, 'sentence': dict } } }
        self.vocabulary: Dict[str, Dict[str, Dict[str, Any]]] = {
            "水果": {
                "apple": {"chinese": "蘋果", "correct": 0, "wrong": 0},
                "banana": {"chinese": "香蕉", "correct": 0, "wrong": 0},
                "orange": {"chinese": "橘子", "correct": 0, "wrong": 0},
            },
            "核心單字": {
                "abandon": {"chinese": "放棄；遺棄", "correct": 0, "wrong": 0},
                "accomplish": {"chinese": "完成；實現", "correct": 0, "wrong": 0},
                "demonstrate": {"chinese": "證明；展示", "correct": 0, "wrong": 0},
                "innovation": {"chinese": "創新；變革", "correct": 0, "wrong": 0},
                "brilliant": {"chinese": "傑出的；燦爛的", "correct": 0, "wrong": 0},
            }
        }

    def all_words(self, category: Optional[str] = None) -> List[Dict[str, Any]]:
        """取得指定分類或所有單字列表"""
        words = []
        for cat, cat_words in self.vocabulary.items():
            if category and cat != category:
                continue
            for en, data in cat_words.items():
                words.append({
                    "english": en,
                    "chinese": data.get("chinese", ""),
                    "category": cat,
                    "correct": data.get("correct", 0),
                    "wrong": data.get("wrong", 0),
                    "sentence": data.get("sentence", None)
                })
        return words

    def add_word(self, english: str, chinese: str, category: str = "預設") -> bool:
        """手動新增單字"""
        en = english.strip().lower()
        zh = chinese.strip()
        cat = category.strip() or "預設"
        if not en:
            return False
        self.vocabulary.setdefault(cat, {})
        if en in self.vocabulary[cat]:
            self.vocabulary[cat][en]["chinese"] = zh
            print(f"⚠️ 單字 '{en}' 已存在，已更新釋義為：{zh}")
        else:
            self.vocabulary[cat][en] = {"chinese": zh, "correct": 0, "wrong": 0}
            print(f"✅ 已新增單字：{en} -> {zh} [{cat}]")
        return True

    def delete_word(self, english: str, category: Optional[str] = None) -> bool:
        """刪除單字"""
        en = english.strip().lower()
        deleted = False
        cats_to_check = [category] if category else list(self.vocabulary.keys())
        for cat in cats_to_check:
            if cat in self.vocabulary and en in self.vocabulary[cat]:
                del self.vocabulary[cat][en]
                deleted = True
                print(f"🗑️ 已自分類 [{cat}] 刪除單字：{en}")
        return deleted

    def import_csv(self, file_path_or_text: str, category: str = "CSV匯入") -> int:
        """匯入 CSV 檔案或 CSV 格式字串"""
        content = ""
        if os.path.exists(file_path_or_text):
            encodings = ["utf-8-sig", "utf-8", "big5", "cp950", "gbk"]
            for enc in encodings:
                try:
                    with open(file_path_or_text, "r", encoding=enc) as f:
                        content = f.read()
                    break
                except UnicodeDecodeError:
                    continue
        else:
            content = file_path_or_text

        lines = [line.strip() for line in content.strip().splitlines() if line.strip()]
        if not lines:
            print("❌ CSV 內容為空。")
            return 0

        delimiter = ","
        if "\t" in lines[0]:
            delimiter = "\t"

        reader = csv.reader(lines, delimiter=delimiter)
        rows = [r for r in reader if r and any(c.strip() for c in r)]
        if not rows:
            return 0

        first_row = [c.strip().lower() for c in rows[0]]
        en_idx, zh_idx = 0, 1
        start_idx = 0

        # 辨識 Header 欄位名稱
        header_candidates_en = ["english", "word", "term", "英文", "單字"]
        header_candidates_zh = ["chinese", "meaning", "translation", "中文", "釋義"]
        found_en = next((i for i, c in enumerate(first_row) if any(h in c for h in header_candidates_en)), None)
        found_zh = next((i for i, c in enumerate(first_row) if any(h in c for h in header_candidates_zh)), None)

        if found_en is not None and found_zh is not None:
            en_idx = found_en
            zh_idx = found_zh
            start_idx = 1
        elif "english" in first_row[0] or "word" in first_row[0]:
            start_idx = 1

        imported_count = 0
        for row in rows[start_idx:]:
            if len(row) > max(en_idx, zh_idx):
                en = row[en_idx].strip()
                zh = row[zh_idx].strip()
            elif len(row) >= 2:
                en = row[0].strip()
                zh = row[1].strip()
            else:
                continue

            if en:
                self.add_word(en, zh, category=category)
                imported_count += 1

        print(f"🎉 成功匯入 {imported_count} 筆單字至分類 [{category}]！")
        return imported_count

    def export_csv(self, filename: str = "vocabulary_export.csv") -> str:
        """匯出單字庫至 CSV 檔案"""
        words = self.all_words()
        with open(filename, "w", encoding="utf-8-sig", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["English", "Chinese", "Category"])
            for w in words:
                writer.writerow([w["english"], w["chinese"], w["category"]])
        print(f"💾 已將 {len(words)} 筆單字匯出至：{filename}")
        return filename

    def show_summary(self):
        """顯示目前單字庫與學習統計總覽"""
        words = self.all_words()
        total_correct = sum(w["correct"] for w in words)
        total_wrong = sum(w["wrong"] for w in words)
        total_attempts = total_correct + total_wrong
        acc = (total_correct / total_attempts * 100) if total_attempts > 0 else 0.0

        print("=" * 50)
        print("          📚 Vocabulary-Go 單字庫總覽")
        print("=" * 50)
        print(f"  單字總數：{len(words)} 個")
        print(f"  答對次數：{total_correct} 次")
        print(f"  答錯次數：{total_wrong} 次")
        print(f"  總體答對率：{acc:.1f}%")
        print("-" * 50)
        print("  分類分佈：")
        for cat, cat_words in self.vocabulary.items():
            print(f"    • {cat}：{len(cat_words)} 個單字")
        print("=" * 50)

vocab_manager = VocabularyManager()
vocab_manager.show_summary()

          📚 Vocabulary-Go 單字庫總覽
  單字總數：8 個
  答對次數：0 次
  答錯次數：0 次
  總體答對率：0.0%
--------------------------------------------------
  分類分佈：
    • 水果：3 個單字
    • 核心單字：5 個單字


In [3]:
class BlankifyEngine:
    """移植自 Vocabulary-Go 的核心挖空與相似度比對演算法"""

    prepositions = [
        "the", "about", "above", "across", "after", "against", "along", "among",
        "around", "at", "before", "behind", "below", "beneath", "beside",
        "between", "beyond", "but", "by", "concerning", "despite", "down",
        "during", "except", "for", "from", "in", "inside", "into", "like",
        "near", "of", "off", "on", "onto", "out", "outside", "over", "past",
        "regarding", "round", "since", "through", "throughout", "to", "toward",
        "under", "underneath", "until", "up", "upon", "with", "within", "without"
    ]

    @classmethod
    def blankify(cls, word: str) -> str:
        """單字挖空演算法核心"""
        # 保留前後標點符號
        m_lead = re.match(r"^([^\w]+)", word)
        m_trail = re.search(r"([^\w]+)$", word)
        lead = m_lead.group(1) if m_lead else ""
        trail = m_trail.group(1) if m_trail else ""
        clean = word.strip(".,!?;:\"'()[]{}~`。，！？；：")

        if not clean:
            return word

        clean_lower = clean.lower()

        if clean_lower in cls.prepositions:
            masked = "___"
        elif len(clean) <= 3:
            masked = "___"
        elif clean_lower.endswith("ed") and len(clean) >= 3:
            masked = clean[0] + "___"
        elif clean_lower.endswith("ing") and len(clean) >= 4:
            masked = clean[0] + "___"
        elif len(clean) >= 4:
            masked = clean[0] + "____" + clean[-1]
        else:
            masked = "___"

        return f"{lead}{masked}{trail}"

    @classmethod
    def check_similarity(cls, phrase1: str, phrase2: str) -> float:
        """使用 SequenceMatcher 比對字串相似度"""
        return SequenceMatcher(None, phrase1.lower(), phrase2.lower()).ratio()

    @classmethod
    def best_match(cls, text: str, target_phrase: str) -> Tuple[str, float, List[int]]:
        """在句子中找出與目標單字/片語最相似的子字串與索引位置"""
        words = text.split()
        target_words = target_phrase.split()
        target_len = len(target_words)

        best_similarity = 0.0
        best_match_str = ""
        best_positions: List[int] = []

        if target_len == 1:
            for i, word in enumerate(words):
                clean_word = word.strip(".,!?;:\"'()[]{}~`。，！？；：")
                sim = cls.check_similarity(clean_word, target_phrase)
                if sim > best_similarity:
                    best_similarity = sim
                    best_match_str = clean_word
                    best_positions = [i]
        else:
            for i in range(len(words) - target_len + 1):
                chunk = " ".join(words[i:i + target_len])
                clean_chunk = " ".join([w.strip(".,!?;:\"'()[]{}~`。，！？；：") for w in words[i:i + target_len]])
                sim = cls.check_similarity(clean_chunk, target_phrase)
                if sim > best_similarity:
                    best_similarity = sim
                    best_match_str = clean_chunk
                    best_positions = list(range(i, i + target_len))

        return best_match_str, best_similarity, best_positions

    @classmethod
    def mask_sentence(cls, sentence: str, target_phrase: str) -> Tuple[str, str, float]:
        """將英文句子中的目標單字進行挖空"""
        words = sentence.split()
        best_str, sim, positions = cls.best_match(sentence, target_phrase)

        # 邊界容錯回退搜尋
        if not positions or sim < 0.4:
            for i, w in enumerate(words):
                if target_phrase.lower() in w.lower():
                    positions = [i]
                    sim = 1.0
                    best_str = w.strip(".,!?;:")
                    break

        if not positions:
            raise ValueError(f"無法在句子中定位單字 '{target_phrase}'")

        for idx in positions:
            if idx < len(words):
                words[idx] = cls.blankify(words[idx])

        masked_sentence = " ".join(words).strip()
        return masked_sentence, best_str, sim

# 演算法效果單元測試
test_sentence = "The dedicated teacher demonstrated an innovative method to inspire students."
test_target = "demonstrated"
masked_s, best_w, similarity = BlankifyEngine.mask_sentence(test_sentence, test_target)

print("===== 演算法挖空測試 =====")
print(f"原句：{test_sentence}")
print(f"目標單字：{test_target}")
print(f"最佳匹配：{best_w} (相似度: {similarity:.2f})")
print(f"挖空結果：{masked_s}")

===== 演算法挖空測試 =====
原句：The dedicated teacher demonstrated an innovative method to inspire students.
目標單字：demonstrated
最佳匹配：demonstrated (相似度: 1.00)
挖空結果：The dedicated teacher d___ an innovative method to inspire students.


In [10]:
class GroqSentenceGenerator:
    """Groq API 小模型例句生成器"""
    API_URL = "https://api.groq.com/openai/v1/chat/completions"

    def __init__(self, api_key: Optional[str] = None, model: str = "qwen/qwen3.8-27b"):
        self.api_key = api_key or os.environ.get("GROQ_API_KEY", "")
        self.model = model

    def set_api_key(self, api_key: str):
        self.api_key = api_key.strip()

    def generate_raw_sentence(self, phrase: str) -> str:
        """向 Groq API 請求例句與繁體中文翻譯"""
        if not self.api_key:
            raise ValueError("請先設定 Groq API Key！")

        # Vocabulary-Go 原始 Prompt
        prompt = (
            f"You are a sentence-making tool. Make *1* short sentence. The sentence must use `{phrase}`. "
            f"Do not use other hard words and do not use Markdown. The sentence should look like a vocabulary test sentence. "
            f"After the sentence, give a **whole** sentence **Traditional Chinese** translation. "
            f"You MUST use Traditional Chinese characters for the translation, not Simplified Chinese. "
            f"Use `|` to separate the English sentence and the Chinese translation."
        )

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": "You are a vocabulary test sentence generator. Output exactly 1 sentence followed by | and Traditional Chinese translation.\向"},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.7,
            "max_tokens": 150
        }

        with httpx.Client(timeout=20.0) as client:
            response = client.post(self.API_URL, headers=headers, json=payload)
            if response.status_code == 401:
                raise PermissionError("Groq API Key 無效或未授權 (401 Unauthorized)。")
            if response.status_code == 429:
                raise TimeoutError("Groq API 請求頻率過高 (429 Rate Limited)，請稍後再試。")
            if response.status_code != 200:
                raise RuntimeError(f"Groq API 回應錯誤 {response.status_code}: {response.text}")

            data = response.json()
            return data["choices"][0]["message"]["content"].strip()

    def create_question(self, english_word: str, chinese_meaning: str = "") -> Optional[Dict[str, Any]]:
        """生成句子、執行演算法挖空並驗證有效性"""
        for attempt in range(1, 4):
            try:
                raw_response = self.generate_raw_sentence(english_word)
                cleaned = raw_response.strip("`'\" \n\r")
                if "|" not in cleaned:
                    continue

                raw_en, raw_zh = cleaned.split("|", 1)
                raw_en = raw_en.strip()
                raw_zh = raw_zh.strip()

                # 轉換為繁體中文
                if cc:
                    try:
                        raw_zh = cc.convert(raw_zh)
                    except Exception:
                        pass

                # 演算法挖空
                masked_en, best_word, sim = BlankifyEngine.mask_sentence(raw_en, english_word)

                # 相似度與格式檢查
                if sim < 0.45:
                    continue

                return {
                    "word_english": english_word,
                    "word_chinese": chinese_meaning,
                    "masked_sentence": masked_en,
                    "raw_sentence": raw_en,
                    "chinese_translation": raw_zh,
                    "best_matched_word": best_word,
                    "similarity": round(sim, 3)
                }
            except Exception as e:
                if attempt == 3:
                    print(f"❌ 為 '{english_word}' 生成題目失敗：{e}")
                time.sleep(1.0)
        return None

groq_gen = GroqSentenceGenerator()
print("✅ GroqSentenceGenerator 引擎初始化完成！")

✅ GroqSentenceGenerator 引擎初始化完成！


In [13]:
class SentenceQuiz:
    """造句挖空互動測驗系統"""

    def __init__(self, vocab_mgr: VocabularyManager, groq_generator: GroqSentenceGenerator):
        self.vocab_mgr = vocab_mgr
        self.groq_gen = groq_generator
        self.questions_cache: List[Dict[str, Any]] = []

    def prepare_questions(self, count: int = 5, category: Optional[str] = None) -> List[Dict[str, Any]]:
        """為測驗準備題目（若未快取則現場透過 Groq 生成）"""
        words = self.vocab_mgr.all_words(category=category)
        if not words:
            print("⚠️ 單字庫中沒有單字，請先新增單字或匯入 CSV！")
            return []

        selected_words = random.sample(words, min(count, len(words)))
        questions = []

        print(f"\n⏳ 正在準備 {len(selected_words)} 題句子挖空題目...")
        for idx, w in enumerate(selected_words, start=1):
            en = w["english"]
            zh = w["chinese"]

            # 檢查單字是否已有生成好的句子快取
            cached_sentence = w.get("sentence")
            if cached_sentence:
                questions.append(cached_sentence)
                print(f"  [{idx}/{len(selected_words)}] 使用快取：{en}")
                continue

            print(f"  [{idx}/{len(selected_words)}] 透過 Groq 生成造句中：{en} ({zh})...")
            q = self.groq_gen.create_question(en, zh)
            if q:
                # 快取回單字庫
                cat = w["category"]
                self.vocab_mgr.vocabulary[cat][en]["sentence"] = q
                questions.append(q)
            else:
                # 若未設定 API Key 或生成失敗，提供離線備用挖空
                fallback_s = f"This is an example sentence featuring the word {en}."
                masked, _, _ = BlankifyEngine.mask_sentence(fallback_s, en)
                fallback_q = {
                    "word_english": en,
                    "word_chinese": zh,
                    "masked_sentence": masked,
                    "raw_sentence": fallback_s,
                    "chinese_translation": f"（範例句）這是一個關於 {zh} 的測驗句子。",
                    "best_matched_word": en,
                    "similarity": 1.0
                }
                questions.append(fallback_q)

        self.questions_cache = questions
        return questions

    def run_quiz(self, question_count: int = 5, category: Optional[str] = None):
        """執行互動式挖空句子測驗"""
        questions = self.prepare_questions(count=question_count, category=category)
        if not questions:
            return

        print("\n" + "=" * 55)
        print("          🚀 Vocabulary-Go 句子挖空測驗開始！")
        print("=" * 55)
        print("提示：根據中文翻譯與挖空線索，輸入正確的英文單字。")

        correct_cnt = 0
        total = len(questions)

        for i, q in enumerate(questions, start=1):
            print(f"\n第 {i} / {total} 題：")
            print(f"【中文翻譯】：{q['chinese_translation']}")
            print(f"【挖空句子】：{q['masked_sentence']}")

            user_input = input("👉 請輸入挖空的單字：").strip().lower()
            target = q["word_english"].strip().lower()

            # 判定驗證
            is_correct = (user_input == target)

            # 更新統計至單字庫
            for cat, cat_dict in self.vocab_mgr.vocabulary.items():
                if q["word_english"] in cat_dict:
                    if is_correct:
                        cat_dict[q["word_english"]]["correct"] += 1
                    else:
                        cat_dict[q["word_english"]]["wrong"] += 1

            if is_correct:
                correct_cnt += 1
                print("🎉 【答對！】太棒了！")
            else:
                print(f"❌ 【答錯！】正確答案是：{q['word_english']}")

            print(f"📖 【完整原句】：{q['raw_sentence']}")
            print("-" * 45)

        accuracy = (correct_cnt / total * 100) if total > 0 else 0.0
        print("\n" + "=" * 55)
        print("                  📊 測驗結算")
        print("=" * 55)
        print(f"  總題數：{total} 題")
        print(f"  答對數：{correct_cnt} 題")
        print(f"  答錯數：{total - correct_cnt} 題")
        print(f"  本次正確率：{accuracy:.1f}%")
        print("=" * 55)

sentence_quiz = SentenceQuiz(vocab_manager, groq_gen)
print("✅ 測驗系統 SentenceQuiz 初始化完成！")

✅ 測驗系統 SentenceQuiz 初始化完成！


In [14]:
!pip install -q gradio pandas

import gradio as gr
import pandas as pd
import os
import random
import tempfile


# =========================================================
# Groq API 驗證
# =========================================================

def verify_groq_key(key):
    key = key.strip()

    if not key:
        return (
            "請輸入 Groq API Key。",
            gr.update(visible=False)
        )

    if not key.startswith("gsk_"):
        return (
            "API Key 格式錯誤。",
            gr.update(visible=False)
        )

    try:
        # 用一個很短的請求實際驗證 API Key
        groq_gen.set_api_key(key)

        test = groq_gen.generate_raw_sentence("apple")

        if test and "|" in test:
            return (
                "API Key 驗證成功。",
                gr.update(visible=True)
            )

        return (
            "API Key 可以連線，但 Groq 回應格式錯誤。",
            gr.update(visible=False)
        )

    except PermissionError:
        return (
            "API Key 無效或未授權。",
            gr.update(visible=False)
        )

    except Exception as e:
        return (
            f"Groq 連線失敗：{e}",
            gr.update(visible=False)
        )


# =========================================================
# 單字庫
# =========================================================

def get_categories():
    words = vocab_manager.all_words()

    categories = sorted(
        set(w["category"] for w in words)
    )

    return ["全部"] + categories


def get_word_table(category="全部"):
    words = vocab_manager.all_words()

    if category != "全部":
        words = [
            w for w in words
            if w["category"] == category
        ]

    data = []

    for w in words:
        en = w["english"]
        cat = w["category"]

        info = vocab_manager.vocabulary[
            cat
        ][en]

        sentence = info.get("sentence")

        correct = info.get("correct", 0)
        wrong = info.get("wrong", 0)

        data.append([
            en,
            w["chinese"],
            cat,
            "有" if sentence else "無",
            correct,
            wrong
        ])

    return pd.DataFrame(
        data,
        columns=[
            "英文",
            "中文",
            "分類",
            "例句",
            "答對",
            "答錯"
        ]
    )


def get_summary(category="全部"):
    words = vocab_manager.all_words()

    if category != "全部":
        words = [
            w for w in words
            if w["category"] == category
        ]

    total = len(words)

    generated = 0
    correct = 0
    wrong = 0

    for w in words:
        info = vocab_manager.vocabulary[
            w["category"]
        ][w["english"]]

        if info.get("sentence"):
            generated += 1

        correct += info.get("correct", 0)
        wrong += info.get("wrong", 0)

    return (
        f"### 單字庫統計\n\n"
        f"單字數：**{total}**\n\n"
        f"已有 AI 例句：**{generated}**\n\n"
        f"尚未有例句：**{total-generated}**\n\n"
        f"累計答對：**{correct}**\n\n"
        f"累計答錯：**{wrong}**"
    )


def refresh_vocab():
    categories = get_categories()

    return (
        gr.update(
            choices=categories,
            value="全部"
        ),
        get_summary("全部"),
        get_word_table("全部")
    )


# =========================================================
# 新增 / 刪除
# =========================================================

def add_word_ui(english, chinese, category):

    english = english.strip()
    chinese = chinese.strip()
    category = category.strip() or "自訂"

    if not english:
        return "請輸入英文單字。"

    if not chinese:
        return "請輸入中文釋義。"

    try:
        vocab_manager.add_word(
            english,
            chinese,
            category
        )

        return f"新增成功：{english} → {chinese}"

    except Exception as e:
        return f"新增失敗：{e}"


def delete_word_ui(english):

    english = english.strip()

    if not english:
        return "請輸入要刪除的單字。"

    try:
        vocab_manager.delete_word(english)

        return f"刪除完成：{english}"

    except Exception as e:
        return f"刪除失敗：{e}"


# =========================================================
# CSV 匯入
# =========================================================

def import_csv_file_ui(file, category):

    if file is None:
        return "請選擇 CSV 檔案。"

    category = category.strip() or "匯入"

    try:
        path = file if isinstance(file, str) else file.name

        vocab_manager.import_csv(
            path,
            category=category
        )

        return "CSV 匯入成功。"

    except Exception as e:
        return f"CSV 匯入失敗：{e}"


def import_csv_text_ui(text, category):

    if not text.strip():
        return "請貼上 CSV 內容。"

    category = category.strip() or "匯入"

    try:
        # 建立暫存 CSV
        temp = tempfile.NamedTemporaryFile(
            mode="w",
            suffix=".csv",
            delete=False,
            encoding="utf-8-sig"
        )

        temp.write(text)
        temp.close()

        vocab_manager.import_csv(
            temp.name,
            category=category
        )

        os.unlink(temp.name)

        return "CSV 匯入成功。"

    except Exception as e:
        return f"CSV 匯入失敗：{e}"


# =========================================================
# CSV 匯出
# =========================================================

def export_csv_ui(filename):

    filename = filename.strip()

    if not filename:
        filename = "vocabulary_export.csv"

    if not filename.endswith(".csv"):
        filename += ".csv"

    try:
        vocab_manager.export_csv(filename)

        if os.path.exists(filename):
            return filename

        return None

    except Exception as e:
        print(e)
        return None


# =========================================================
# AI 批次生成
# =========================================================

def generate_all_sentences_ui():

    if not groq_gen.api_key:
        return "請先設定 Groq API Key。"

    words = vocab_manager.all_words()

    if not words:
        return "單字庫目前沒有單字。"

    success = 0
    skipped = 0
    failed = 0

    output = []

    for i, w in enumerate(words, start=1):

        en = w["english"]
        zh = w["chinese"]
        cat = w["category"]

        info = vocab_manager.vocabulary[
            cat
        ][en]

        # 已經有例句就跳過
        if info.get("sentence"):
            skipped += 1

            output.append(
                f"[{i}/{len(words)}] "
                f"{en}：已有例句，跳過"
            )

            continue

        try:

            q = groq_gen.create_question(
                en,
                zh
            )

            if q:

                info["sentence"] = q

                success += 1

                output.append(
                    f"[{i}/{len(words)}] "
                    f"{en} → {q['masked_sentence']}"
                )

            else:

                failed += 1

                output.append(
                    f"[{i}/{len(words)}] "
                    f"{en}：生成失敗"
                )

        except Exception as e:

            failed += 1

            output.append(
                f"[{i}/{len(words)}] "
                f"{en}：{e}"
            )

    return (
        f"## 生成完成\n\n"
        f"成功：{success}\n\n"
        f"跳過：{skipped}\n\n"
        f"失敗：{failed}\n\n"
        f"---\n\n"
        + "\n".join(output)
    )


# =========================================================
# 測驗系統
# =========================================================

quiz_state = {
    "questions": [],
    "current": 0,
    "correct": 0,
    "category": None
}


def prepare_quiz_ui(count, category):

    words = vocab_manager.all_words(
        category=None if category == "全部"
        else category
    )

    if not words:
        return (
            "目前沒有可用的單字。",
            "",
            "",
            gr.update(visible=False),
            gr.update(visible=False)
        )

    count = int(count)

    selected = random.sample(
        words,
        min(count, len(words))
    )

    questions = []

    for w in selected:

        en = w["english"]
        zh = w["chinese"]
        cat = w["category"]

        info = vocab_manager.vocabulary[
            cat
        ][en]

        q = info.get("sentence")

        if not q:

            q = groq_gen.create_question(
                en,
                zh
            )

            if q:
                info["sentence"] = q

        # Groq 失敗時使用 fallback
        if not q:

            fallback = (
                f"This is an example sentence "
                f"featuring the word {en}."
            )

            masked, best, sim = (
                BlankifyEngine.mask_sentence(
                    fallback,
                    en
                )
            )

            q = {
                "word_english": en,
                "word_chinese": zh,
                "masked_sentence": masked,
                "raw_sentence": fallback,
                "chinese_translation":
                    f"這是一個使用「{zh}」的例句。",
                "best_matched_word": best,
                "similarity": sim
            }

        questions.append(q)

    quiz_state["questions"] = questions
    quiz_state["current"] = 0
    quiz_state["correct"] = 0
    quiz_state["category"] = category

    return show_quiz_question()


def show_quiz_question():

    questions = quiz_state["questions"]
    current = quiz_state["current"]

    if current >= len(questions):

        total = len(questions)
        correct = quiz_state["correct"]

        accuracy = (
            correct / total * 100
            if total > 0 else 0
        )

        return (
            f"## 測驗完成\n\n"
            f"總題數：**{total}**\n\n"
            f"答對：**{correct}**\n\n"
            f"答錯：**{total-correct}**\n\n"
            f"正確率：**{accuracy:.1f}%**",
            "",
            "",
            gr.update(visible=False),
            gr.update(visible=True)
        )

    q = questions[current]

    progress = (
        f"## 第 {current+1} / {len(questions)} 題\n\n"
        f"中文：{q['chinese_translation']}\n\n"
        f"句子：\n\n"
        f"### {q['masked_sentence']}"
    )

    return (
        progress,
        "",
        "",
        gr.update(visible=True),
        gr.update(visible=False)
    )


def submit_answer(answer):

    questions = quiz_state["questions"]
    current = quiz_state["current"]

    if not questions:
        return (
            "請先開始測驗。",
            "",
            gr.update(visible=False)
        )

    if current >= len(questions):
        return (
            show_quiz_question()[0],
            "",
            gr.update(visible=False)
        )

    answer = answer.strip().lower()

    q = questions[current]

    target = q["word_english"].strip().lower()

    is_correct = answer == target

    # 更新統計
    for cat, cat_dict in vocab_manager.vocabulary.items():

        if target in [
            x.lower() for x in cat_dict.keys()
        ]:

            # 找到原始 key
            real_key = next(
                x for x in cat_dict
                if x.lower() == target
            )

            if is_correct:
                cat_dict[real_key]["correct"] = (
                    cat_dict[real_key].get(
                        "correct", 0
                    ) + 1
                )
            else:
                cat_dict[real_key]["wrong"] = (
                    cat_dict[real_key].get(
                        "wrong", 0
                    ) + 1
                )

            break

    if is_correct:

        quiz_state["correct"] += 1

        result = (
            "### 答對\n\n"
            f"答案：**{q['word_english']}**"
        )

    else:

        result = (
            "### 答錯\n\n"
            f"你的答案：**{answer or '未作答'}**\n\n"
            f"正確答案：**{q['word_english']}**"
        )

    result += (
        f"\n\n完整句子：\n"
        f"{q['raw_sentence']}"
    )

    quiz_state["current"] += 1

    return (
        result,
        "",
        gr.update(visible=True)
    )


def next_question():

    if quiz_state["current"] >= len(
        quiz_state["questions"]
    ):

        result = show_quiz_question()

        return (
            result[0],
            gr.update(visible=False),
            gr.update(visible=True)
        )

    q = quiz_state["questions"][
        quiz_state["current"]
    ]

    progress = (
        f"## 第 "
        f"{quiz_state['current']+1}"
        f" / {len(quiz_state['questions'])} 題\n\n"
        f"中文：{q['chinese_translation']}\n\n"
        f"### {q['masked_sentence']}"
    )

    return (
        progress,
        gr.update(visible=True),
        gr.update(visible=False)
    )


# =========================================================
# Blankify
# =========================================================

def blankify_ui(sentence, word):

    sentence = sentence.strip()
    word = word.strip()

    if not sentence or not word:
        return "請輸入句子和單字。"

    try:

        masked, best, sim = (
            BlankifyEngine.mask_sentence(
                sentence,
                word
            )
        )

        return (
            f"最佳匹配詞：{best}\n"
            f"相似度：{sim:.2f}\n\n"
            f"挖空結果：\n{masked}"
        )

    except Exception as e:

        return f"挖空失敗：{e}"


# =========================================================
# Gradio APP
# =========================================================

with gr.Blocks(
    title="Vocabulary-Go"
) as app:

    # =====================================================
    # LOGIN
    # =====================================================

    with gr.Column() as login_page:

        gr.Markdown(
            """
            # Vocabulary-Go

            ## Groq API 設定
            """
        )

        api_key = gr.Textbox(
            label="Groq API Key",
            placeholder="gsk_...",
            type="password"
        )

        login_button = gr.Button(
            "驗證並進入",
            variant="primary"
        )

        login_status = gr.Markdown()


    # =====================================================
    # MAIN MENU
    # =====================================================

    with gr.Column(
        visible=False
    ) as main_page:

        gr.Markdown("# Vocabulary-Go")

        # =================================================
        # 單字庫
        # =================================================

        with gr.Tab("單字庫"):

            with gr.Row():

                category_dropdown = gr.Dropdown(
                    choices=get_categories(),
                    value="全部",
                    label="查看分類"
                )

                refresh_button = gr.Button(
                    "重新整理"
                )

            summary_output = gr.Markdown()

            word_table = gr.Dataframe(
                headers=[
                    "英文",
                    "中文",
                    "分類",
                    "例句",
                    "答對",
                    "答錯"
                ],
                interactive=False,
                wrap=True
            )

            # 新增
            gr.Markdown("### 新增單字")

            with gr.Row():

                add_en = gr.Textbox(
                    label="英文"
                )

                add_zh = gr.Textbox(
                    label="中文"
                )

                add_cat = gr.Textbox(
                    label="分類",
                    value="自訂"
                )

            add_button = gr.Button(
                "新增單字"
            )

            add_result = gr.Markdown()

            add_button.click(
                add_word_ui,
                [
                    add_en,
                    add_zh,
                    add_cat
                ],
                add_result
            )

            # 刪除
            gr.Markdown("### 刪除單字")

            delete_en = gr.Textbox(
                label="英文單字"
            )

            delete_button = gr.Button(
                "刪除"
            )

            delete_result = gr.Markdown()

            delete_button.click(
                delete_word_ui,
                delete_en,
                delete_result
            )

        # =================================================
        # 匯入 / 匯出
        # =================================================

        with gr.Tab("匯入 / 匯出"):

            gr.Markdown("## 匯入 CSV")

            import_category = gr.Textbox(
                label="匯入分類",
                value="匯入"
            )

            csv_file = gr.File(
                label="CSV 檔案",
                file_types=[".csv"]
            )

            import_file_button = gr.Button(
                "匯入 CSV"
            )

            import_file_result = gr.Markdown()

            import_file_button.click(
                import_csv_file_ui,
                [
                    csv_file,
                    import_category
                ],
                import_file_result
            )

            gr.Markdown("### 或直接貼上 CSV")

            csv_text = gr.Textbox(
                label="CSV 內容",
                lines=8,
                placeholder=(
                    "english,chinese\n"
                    "apple,蘋果\n"
                    "banana,香蕉"
                )
            )

            import_text_button = gr.Button(
                "匯入文字"
            )

            import_text_result = gr.Markdown()

            import_text_button.click(
                import_csv_text_ui,
                [
                    csv_text,
                    import_category
                ],
                import_text_result
            )

            gr.Markdown("---")

            gr.Markdown("## 匯出 CSV")

            export_name = gr.Textbox(
                label="檔名",
                value="vocabulary_export.csv"
            )

            export_button = gr.Button(
                "匯出"
            )

            export_file = gr.File(
                label="下載檔案"
            )

            export_button.click(
                export_csv_ui,
                export_name,
                export_file
            )

        # =================================================
        # AI 例句
        # =================================================

        with gr.Tab("AI 例句"):

            gr.Markdown(
                """
                ## 批次生成例句

                已經有例句的單字會自動跳過。
                """
            )

            generate_button = gr.Button(
                "開始生成全部例句",
                variant="primary"
            )

            generate_output = gr.Textbox(
                label="生成結果",
                lines=18
            )

            generate_button.click(
                generate_all_sentences_ui,
                outputs=generate_output
            )

        # =================================================
        # 測驗
        # =================================================

        with gr.Tab("開始測驗"):

            with gr.Row():

                quiz_category = gr.Dropdown(
                    choices=get_categories(),
                    value="全部",
                    label="分類"
                )

                quiz_count = gr.Slider(
                    minimum=1,
                    maximum=20,
                    value=5,
                    step=1,
                    label="題數"
                )

            start_quiz_button = gr.Button(
                "開始測驗",
                variant="primary"
            )

            quiz_question = gr.Markdown(
                "請選擇題數後開始測驗。"
            )

            quiz_answer = gr.Textbox(
                label="你的答案",
                placeholder="輸入英文單字"
            )

            submit_button = gr.Button(
                "提交答案"
            )

            quiz_result = gr.Markdown()

            next_button = gr.Button(
                "下一題",
                visible=False
            )

            restart_button = gr.Button(
                "重新開始",
                visible=False
            )

            start_quiz_button.click(
                prepare_quiz_ui,
                [
                    quiz_count,
                    quiz_category
                ],
                [
                    quiz_question,
                    quiz_result,
                    quiz_answer,
                    submit_button,
                    restart_button
                ]
            )

            submit_button.click(
                submit_answer,
                quiz_answer,
                [
                    quiz_result,
                    quiz_answer,
                    next_button
                ]
            )

            next_button.click(
                next_question,
                outputs=[
                    quiz_question,
                    submit_button,
                    restart_button
                ]
            )

            restart_button.click(
                prepare_quiz_ui,
                [
                    quiz_count,
                    quiz_category
                ],
                [
                    quiz_question,
                    quiz_result,
                    quiz_answer,
                    submit_button,
                    restart_button
                ]
            )

        # =================================================
        # 挖空工具
        # =================================================

        with gr.Tab("挖空工具"):

            sentence_input = gr.Textbox(
                label="英文句子",
                placeholder=(
                    "I ate an apple yesterday."
                )
            )

            word_input = gr.Textbox(
                label="要挖空的單字",
                placeholder="apple"
            )

            blank_button = gr.Button(
                "執行挖空"
            )

            blank_output = gr.Textbox(
                label="結果",
                lines=6
            )

            blank_button.click(
                blankify_ui,
                [
                    sentence_input,
                    word_input
                ],
                blank_output
            )


    # =====================================================
    # Login Event
    # =====================================================

    login_button.click(
        verify_groq_key,
        api_key,
        [
            login_status,
            main_page
        ]
    )


    # =====================================================
    # Vocabulary Events
    # =====================================================

    category_dropdown.change(
        get_summary,
        category_dropdown,
        summary_output
    )

    category_dropdown.change(
        get_word_table,
        category_dropdown,
        word_table
    )

    refresh_button.click(
        refresh_vocab,
        outputs=[
            category_dropdown,
            summary_output,
            word_table
        ]
    )


app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1d89bbb5e5caa87eb1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
